[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# loc and iloc


## What you will be able to do

Pick rows and columns out of a table by label with `loc` and by position with `iloc`, and know which
of the two a question needs, including after the table has been sorted or filtered.

## The idea

### The problem

Square brackets on a DataFrame select columns, as the **Selecting Columns** notebook showed, and a
mask in square brackets selects rows. Neither one answers "give me row 5", "give me the reading at
noon on 2 March" or "give me the first three rows and the second column". Those questions name
rows, and a row can be named by its label in the index or by where it sits in the table.

The two coincide in a freshly built table, whose index is 0, 1, 2 and so on, so row 5 is both the
row labeled 5 and the sixth row. That is why code written against a new table often mixes them up
and still works. Sort the table by temperature and the row labeled 5 moves far down the table, while
the sixth row is now a cold night in January. Index the table by station and there is no row labeled
5 at all. Code that meant one and wrote the other then returns the wrong row, or raises an error
that does not say which of the two was meant.

### What loc and iloc are

> **`loc`** selects rows and columns by **label**: `df.loc[row_labels, column_labels]`, where a label
> is whatever the index or the columns hold, and a slice of labels includes both of its ends.
> **`iloc`** selects rows and columns by **position**: `df.iloc[row_positions, column_positions]`,
> where positions count from 0, and a slice of positions includes its start and excludes its end, as
> a slice of a Python list does.

### Why it works that way

A pandas index can hold numbers, station names, times or anything else. A label lookup needs no order at
all, so `loc` answers "which row is called this?" whatever order the rows are in. A position lookup
needs only the order, so `iloc` answers "which row is third?" whatever the labels are. Keeping the two
in separate methods means a reader of the code can tell which question was asked.

The slice rules differ for the same reason. Labels need not be numbers, so a slice of labels such as
`"Bergen":"Oslo"` cannot stop one before its end, because there is no label "one before Oslo". `loc`
includes the end it names. Positions are numbers, and `iloc` follows Python's rule, where `0:3` means
three rows.

Both take a row part and a column part separated by a comma, and each part can be a single label or
position, a list or a slice. `loc` also takes a mask. `iloc` takes one only as a plain array,
`mask.to_numpy()`, and raises an error when given the Series itself. A single row comes back as a
Series, several rows as a DataFrame, and a single row and column as one value.

### Where this shows up

`loc` and `iloc` come in wherever particular rows are needed, such as the reading at one hour, the
first rows after a sort, or the rows a mask keeps. The **NumPy, Deep Dive** guide picked values out
of an array by position, which is what `iloc` does for a table. **The Index** notebook gives the
readings an index of times and of stations, where `loc` finds rows by time and by station, and the
**Boolean Filtering** notebook uses `loc` with a mask.

### What this notebook covers

- a row and a single value with `loc` and with `iloc`
- what changes for each of them after a sort
- slices of labels and of positions, and which end each one includes
- rows and columns together, and rows chosen by a mask
- `loc` on an index of times and on an index of stations
- five errors, among them a slice that returns one row more than the same slice with `iloc`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd

readings = pd.DataFrame(
    {"celsius": [8.2, 6.1, 7.9]},
    index=["Bergen", "Oslo", "Tromso"],
)
print(readings.loc["Oslo", "celsius"])
print(readings.iloc[1, 0])
print(readings.loc["Bergen":"Oslo"])
print(readings.iloc[0:1])
```

```
6.1
6.1
        celsius
Bergen      8.2
Oslo        6.1
        celsius
Bergen      8.2
```

`loc["Oslo", "celsius"]` named the row and the column, and `iloc[1, 0]` gave their positions, the
second row and the first column, and both found 6.1. The slice of labels from Bergen to Oslo kept
Oslo, and the slice of positions from 0 to 1 stopped before position 1.

## Setup

One library, three helpers for building the readings, and two tables.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings

`year_of_readings` makes a reading for every hour of 2025 at the four stations in `STATIONS`, by the
same formula as the **sqlite3, Deep Dive** guide, with Svalbard's 24 readings for 2 March missing.
`readings` is the table of all 35,040, with the columns `station`, `time` and `celsius`, and an index
from 0 to 35039. `stations` holds each station's latitude, Kirkenes included.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### A row by label and by position

In [2]:
print(readings.loc[0])
print()
print(readings.iloc[0])

station                 Bergen
time       2025-01-01 00:00:00
celsius                   -3.6
Name: 0, dtype: object

station                 Bergen
time       2025-01-01 00:00:00
celsius                   -3.6
Name: 0, dtype: object


In this table they agree. `loc[0]` found the row labeled 0 and `iloc[0]` the first row, and those are
the same row, Bergen at midnight on 1 January. A single row comes back as a Series whose index is the
column names, and whose `Name` is the row's label.

### A single value

A row and a column together give one value:

In [3]:
print(readings.loc[5, "celsius"])
print(readings.iloc[5, 2])

-5.3
-5.3


`loc` named the column, `celsius`, and `iloc` gave its position, 2, the third column. The name is the
safer of the two, because adding a column at the front of the table moves every position after it.

### What a sort does to loc and iloc

Sorting by temperature, with the time as a second key so that tied readings keep a fixed order, puts
the coldest readings first:

In [4]:
coldest_first = readings.sort_values(["celsius", "time"])
coldest_first.head(3)

,station,time,celsius
1070,Svalbard,2025-01-12 03:00:00,-17.3
1546,Svalbard,2025-01-17 02:00:00,-17.2
594,Svalbard,2025-01-07 04:00:00,-17.1


Each row kept its label, so the index is no longer 0, 1, 2. The first row is now the one labeled 1070,
Svalbard at 03:00 on 12 January. `loc` and `iloc` ask different questions of this table:

In [5]:
print(coldest_first.loc[0])
print()
print(coldest_first.iloc[0])
print()
print("position of the row labeled 0:", coldest_first.index.get_loc(0))

station                 Bergen
time       2025-01-01 00:00:00
celsius                   -3.6
Name: 0, dtype: object

station               Svalbard
time       2025-01-12 03:00:00
celsius                  -17.3
Name: 1070, dtype: object

position of the row labeled 0: 6855


`loc[0]` still found the row labeled 0, Bergen at midnight, which the sort moved to position 6,855.
`index.get_loc` gives a label's position. `iloc[0]` found the first row of the table as it is, the
coldest reading of the year. Which one is right depends on the question. "This particular reading"
names a label, and "the top of the list" names a position.

### Slices of labels and of positions

In [6]:
print(readings.loc[0:2, "celsius"])
print()
print(readings.iloc[0:2, 2])

0    -3.6
1    -3.5
2   -14.6
Name: celsius, dtype: float64

0   -3.6
1   -3.5
Name: celsius, dtype: float64


The slice of labels `0:2` kept the rows labeled 0, 1 and 2, three rows, because `loc` includes the
end it names. The slice of positions `0:2` kept the first two rows, because `iloc` stops before its
end. Negative positions count from the end, so `readings.iloc[-1]` is the last row.

### Rows and columns together

Each part can be a list:

In [7]:
readings.loc[[0, 1, 35039], ["station", "celsius"]]

,station,celsius
0,Bergen,-3.6
1,Oslo,-3.5
35039,Tromso,-6.5


In [8]:
readings.iloc[[0, 1, -1], [0, 2]]

,station,celsius
0,Bergen,-3.6
1,Oslo,-3.5
35039,Tromso,-6.5


The same three rows and two columns, chosen by label and then by position. `-1` reached the last row
by position, and its label, 35039, shows it is the same row as in the `loc` version.

### Rows chosen by a mask

A mask in the row part keeps the rows where it is `True`, and the column part chooses what to show of
them:

In [9]:
readings.loc[readings["celsius"] > 20.5, ["station", "time", "celsius"]]

,station,time,celsius
18108,Bergen,2025-07-08 15:00:00,20.6
18200,Bergen,2025-07-09 14:00:00,20.6
18584,Bergen,2025-07-13 14:00:00,20.6
18880,Bergen,2025-07-16 16:00:00,20.7
19264,Bergen,2025-07-20 16:00:00,20.6
19356,Bergen,2025-07-21 15:00:00,20.8
19740,Bergen,2025-07-25 15:00:00,20.6
19832,Bergen,2025-07-26 14:00:00,20.6


Eight readings above 20.5 degrees, all of them Bergen's, in July. The **Boolean Filtering** notebook
covers masks, and `loc` is the way to apply one and choose columns in the same step.

### loc on an index of times

`set_index("time")` makes the times the index, so `loc` can find rows by their time. pandas reads a
time written as text:

In [10]:
by_time = readings.set_index("time")
by_time.loc["2025-03-02 12:00"]

,station,celsius
time,,
2025-03-02 12:00:00,Bergen,3.2
2025-03-02 12:00:00,Oslo,1.6
2025-03-02 12:00:00,Svalbard,NaN
2025-03-02 12:00:00,Tromso,-1.6


Four rows share the label for noon on 2 March, one for each station, and Svalbard's reading is
missing, as it is all that day. A slice of times includes both ends, as every `loc` slice does:

In [11]:
by_time.loc["2025-03-02 00:00":"2025-03-02 01:00"]

,station,celsius
time,,
2025-03-02 00:00:00,Bergen,-1.3
2025-03-02 00:00:00,Oslo,-2.9
2025-03-02 00:00:00,Svalbard,NaN
2025-03-02 00:00:00,Tromso,-4.4
2025-03-02 01:00:00,Bergen,-1.5
2025-03-02 01:00:00,Oslo,-3.1
2025-03-02 01:00:00,Svalbard,NaN
2025-03-02 01:00:00,Tromso,-6.3


Midnight and 01:00, both included, for all four stations. The **Dates and Times** notebook does more
with an index of times.

### loc on an index of stations

In [12]:
by_station = readings.set_index("station")

oslo = by_station.loc["Oslo"]
print(type(oslo).__name__, oslo.shape)
print(by_station.iloc[1])

DataFrame (8760, 2)
time       2025-01-01 00:00:00
celsius                   -3.5
Name: Oslo, dtype: object


The label `Oslo` belongs to 8,760 rows, so `loc["Oslo"]` returned a DataFrame of all of them. `iloc[1]`
still means the second row, which is Oslo at midnight on 1 January. **The Index** notebook covers
indexes whose labels repeat.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/07-loc-and-iloc-solutions.ipynb).

**1.** Print the reading in row 100 of `readings`, by label with `loc` and by position with `iloc`, and check that they agree.

In [13]:
# your code here

**2.** Sort `readings` from warmest to coldest with `sort_values(["celsius", "time"], ascending=[False, True])`, and print the warmest row with `iloc`.

In [14]:
# your code here

**3.** Show the last three rows of `readings` with `iloc` and a negative slice.

In [15]:
# your code here

**4.** Show the `station` and `celsius` columns for the readings below -17 degrees, with `loc` and a mask.

In [16]:
# your code here

**5.** Make a table indexed by time, as `by_time` is, and show the readings at 15:00 on 21 July 2025.

In [17]:
# your code here

## Common errors

### KeyError: 0

In [18]:
by_station.loc[0]

KeyError: 0

`by_station` is indexed by station name, so it has no row labeled 0. `loc` looks only at labels, and
reports the label it could not find. The first row by position is `by_station.iloc[0]`.

### IndexError: single positional indexer is out-of-bounds

In [19]:
readings.iloc[35040]

IndexError: single positional indexer is out-of-bounds

Positions run from 0 to `len(readings) - 1`, 35039 here, so 35040 is one past the end. "Single
positional indexer" means one position given to `iloc`. For the last row, write `readings.iloc[-1]`.

### TypeError: Cannot index by location index with a non-integer key

In [20]:
readings.iloc["celsius"]

TypeError: Cannot index by location index with a non-integer key

`iloc` takes positions, and a column's name is a label. pandas calls positions "location", hence the
wording. Use `readings.loc[:, "celsius"]`, or `readings["celsius"]` for the whole column. With `iloc`,
the column's position is 2: `readings.iloc[:, 2]`.

### NotImplementedError: iLocation based boolean indexing on an integer type is not available

In [21]:
warm = readings["celsius"] > 20.5

readings.iloc[warm]

NotImplementedError: iLocation based boolean indexing on an integer type is not available

A mask is a Series of `True` and `False` with the table's index, so it carries labels, and `iloc` deals
only in positions. The message names the mask's index, which here holds integers, the labels 0 to
35039. `loc` takes the mask as it is. `iloc` takes it once it is a plain array of
`True` and `False`, with no index:

In [22]:
print(len(readings.loc[warm]), len(readings.iloc[warm.to_numpy()]))

8 8


Both keep the same eight readings. `loc` is the usual choice, since it can name the columns in the
same step.

### No error, and three rows from loc[0:2] where iloc[0:2] gave two

In [23]:
print(len(readings.loc[0:2]), len(readings.iloc[0:2]))

3 2


Both are valid, and they disagree by one row. `loc` included the row labeled 2, and `iloc` stopped
before position 2. Code that takes "the first n rows" with `loc[0:n]` gets n + 1 of them. For a number of
rows from the top, use `iloc[0:n]` or `head(n)`.

## Recap

- Select by label with `loc` and by position with `iloc`, with the rows before the comma and the
  columns after it
- Tell which one a question needs, and what each returns after a sort
- Remember that a `loc` slice includes its end and an `iloc` slice stops before it
- Combine a mask and a list of columns in one `loc`
- Find rows by time or by station with `loc` on a table indexed by either

## What is next

The **Boolean Filtering** notebook builds the masks that `loc` used here, and combines several
conditions without the mistakes Python's operators invite.

---

&#8592; **Previous:** [Selecting Columns](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/06-selecting-columns.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Boolean Filtering](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/08-boolean-filtering.ipynb) &#8594;
